### LIBRARIES AND PACAKAGES LOADING

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")


### EDA

In [ ]:
df = pd.read_csv(r"C:\Users\RAMESH\OneDrive\Desktop\NutriClass\data\raw\synthetic_food_dataset_imbalanced.csv")
df.head()


In [ ]:
df.info()
df.describe()

In [ ]:
df.isnull().sum()

df.duplicated().sum()


In [ ]:
df = df.drop_duplicates()


In [ ]:
df.columns


In [ ]:
df = df.fillna(df.mean(numeric_only=True))
df.isnull().sum()


In [ ]:
sns.countplot(x=df['Food_Name'])
plt.xticks(rotation=90)
plt.title("Class Distribution")
plt.show()



In [ ]:
df.boxplot(figsize=(14,6))
plt.xticks(rotation=45)
plt.title("Boxplot of Numeric Features")
plt.show()


In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
num_cols

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


### DATA PREPROCESSING

In [ ]:
X = df.drop("Food_Name", axis=1)
y = df["Food_Name"]


In [ ]:
df_encoded = pd.get_dummies(df, columns=['Meal_Type', 'Preparation_Method'], drop_first=True)
df_encoded.head()

In [ ]:
X = df_encoded.drop("Food_Name", axis=1)
y = df_encoded["Food_Name"]


In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [ ]:
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)


#### TRAIN-TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### MODEL TRAINING

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": xgb.XGBClassifier(eval_metric='mlogloss')
}

results = {}

for name, model in models.items():
    print("="*60)
    print(name)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    
    print("Accuracy:", acc)
    print("\nClassification Report:")
    print(classification_report(y_test, pred))
    
    results[name] = acc


### CONFUSION METRICS

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for name, model in models.items():
    plt.figure(figsize=(6,4))
    disp = ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test)
    plt.title(f"Confusion Matrix — {name}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


### FEATURE IMPORTANCE

In [ ]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("Best Model:", best_model_name)

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    plt.figure(figsize=(10,6))
    plt.bar(X.columns, importances)
    plt.xticks(rotation=90)
    plt.title("Feature Importance")
    plt.show()
else:
    print("This model does not support feature_importances_.")

## FEATURE IMPORTANCE OF TREE MODELS

# Loop through all models and extract feature importances ONLY for tree-based models

for name, model in models.items():
    print("="*60)
    print(f"Feature Importance for: {name}")
    
    # Check if model supports feature_importances_
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        feature_names = X.columns
        
        # Sort importances
        indices = np.argsort(importances)[::-1]
        
        # Plot
        plt.figure(figsize=(10, 6))
        plt.title(f"Feature Importance - {name}")
        plt.bar(range(len(importances)), importances[indices])
        plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ Feature importance not supported for model: {name}")
